# AgriSmart AI — ICAR Rice & Maize Model Training Pipeline
**Dataset**: ICAR Crop Disease and Insect-pest Image Dataset for Rice and Maize (ICAR-IASRI, Govt of India)
**Portal**: https://aikosh.indiaai.gov.in/home/datasets/details/crop_disease_and_insect_pest_image_dataset_for_rice_and_maize.html
**Target Architecture**: EfficientNet-B0 Transfer Learning (PyTorch)

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Mount Google Drive or Upload Dataset ZIP
from google.colab import drive
drive.mount('/content/drive')

# If you uploaded the icar_dataset.zip to Drive:
!mkdir -p /content/dataset
# !unzip -q "/content/drive/MyDrive/icar_dataset.zip" -d /content/dataset

In [ ]:
# 3. PyTorch Model Training with Macro-F1 Evaluation
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# Image Transformations
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder('/content/dataset/train', transform=train_tf)
val_dataset = datasets.ImageFolder('/content/dataset/val', transform=val_tf)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

class_names = train_dataset.classes
num_classes = len(class_names)
print(f"Classes ({num_classes}): {class_names}")

In [ ]:
# 4. EfficientNet-B0 Model Setup
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# 5. Training Loop
EPOCHS = 10
best_f1 = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    scheduler.step()
    train_acc = correct / total
    
    # Validation
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Acc: {train_acc:.4f} | Val Macro-F1: {macro_f1:.4f}")
    
    if macro_f1 > best_f1:
        best_f1 = macro_f1
        torch.save(model, "/content/model_weights.pt")
        print(f"  --> Saved Best Model Checkpoint (Macro-F1: {best_f1:.4f})")

In [ ]:
# 6. Generate Confusion Matrix & Final Report
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('ICAR Rice & Maize Disease Confusion Matrix')
plt.ylabel('True Class')
plt.xlabel('Predicted Class')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=300)
plt.show()

# 7. Download Model Weights for AgriSmart AI
from google.colab import files
files.download('/content/model_weights.pt')
print("Upload 'model_weights.pt' into your AgriSmart AI 'model/' folder!")